# Exercise 6: PCA for airfoils

Can we describe an image of a flow using only a few numbers? We will use PCA to find spatial patterns in the speed field around an airfoil, then combine them to reconstruct individual snapshots.

The supplied dataset contains **200 snapshots**, each with 207 rows and 500 columns. Each pixel records the **speed** (velocity magnitude), not its direction. The white region is the airfoil and is excluded from the analysis. The data come from a simulation of the NACA 0012 airfoil [1].

![Short preview of the flow](velocity_animation.gif)

The preview is a short excerpt from the original simulation. The PCA calculations below use the supplied 200-snapshot dataset.


Sections 1–5 cover the **core material**. The extensions at the end explore the same ideas further.

In the lectures, PCA is introduced as a way to reduce dimensionality by projecting data onto the directions of largest variation. Here we give this geometry a concrete meaning: principal components become spatial patterns in a flow, and their coefficients describe how those patterns vary between snapshots. We explore how this representation supports compression, visualization and reconstruction, and how it can help identify a strong external disturbance.

### 1. Understand the data (core)

**Setup — provided.** Open the notebook from the `exercise_06_pca` folder, keeping the `data` subfolder beside it. This cell loads the arrays and orients the images consistently with the preview.

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

data_path = Path("data/velocities_small.npz")
if not data_path.is_file():
    raise FileNotFoundError(
        "Open the notebook from the exercise_06_pca folder, "
        "with data/velocities_small.npz inside it. Extract the Moodle ZIP first."
    )

with np.load(data_path, allow_pickle=False) as data_file:
    vel_grids = np.flip(data_file["mag"], axis=1)
    foil_grid = np.flip(data_file["mask_grid"], axis=0)

nt, NY, NX = vel_grids.shape
NF = int(foil_grid.sum())  # Number of pixels inside the airfoil.
# Use double precision for the SVD and projection comparisons.
data = vel_grids[:, ~foil_grid].astype(float)  # Rows: snapshots; columns: unmasked pixels.
d = data.shape[1]

print(f"Loaded {nt} snapshots, each {NY} × {NX} pixels.")
print(f"Excluded {NF} airfoil pixels. Data matrix shape: {data.shape}.")

Loaded 200 snapshots, each 207 × 500 pixels.
Excluded 1288 airfoil pixels. Data matrix shape: (200, 102212).


**1.a)**

In the course notes the data matrix has shape $n\times d$. Here $n=T=\mathtt{nt}$ is the number of snapshots and $d$ is the number of unmasked pixels. Each snapshot $\mathbf{x}_t\in\mathbb{R}^d$ occupies one **row** of the matrix.

**Plotting helpers.** Complete the TODOs inside the plotting cells: choose the arrays and indices to display. The figure formatting is supplied. Here, restore a pixel vector to the image grid and display the first snapshot.

In [ ]:
speed_cmap = plt.get_cmap("viridis").copy()
speed_cmap.set_bad("white")
mode_cmap = plt.get_cmap("RdBu_r").copy()
mode_cmap.set_bad("white")

def visualize_2d(x):
    """Turn a vector of d unmasked pixels into a (NY, NX) image."""
    image = np.full((NY, NX), np.nan)
    image[~foil_grid] = ...  # TODO: fill the unmasked pixels with the input vector
    return image

def show_field(ax, values, title, signed=False, limits=None):
    if limits is None:
        if signed:
            bound = float(np.max(np.abs(values)))
            limits = (-bound, bound)
        else:
            limits = (float(np.min(values)), float(np.max(values)))
    image = ax.imshow(
        visualize_2d(values),
        cmap=mode_cmap if signed else speed_cmap,
        vmin=limits[0], vmax=limits[1],
    )
    ax.set_title(title)
    ax.set_xlabel("Horizontal pixel")
    ax.set_ylabel("Vertical pixel")
    return image

fig, ax = plt.subplots(figsize=(8, 3))
image = show_field(ax, ..., "First snapshot")  # TODO: select the first snapshot from data
fig.colorbar(image, ax=ax, label="Speed")
plt.show()

**1.b)** Compute the mean speed at each pixel and subtract it from every snapshot. Which axis should you average over?

$$\bar{\mathbf{x}}=\frac{1}{T}\sum_{t=1}^{T}\mathbf{x}_t,
\qquad (X_c)_{tj}=x_{t,j}-\bar{x}_j.$$

The centered matrix $X_c$ still has shape $T\times d$.

**A choice of preprocessing.** The notes also divide each feature by its standard deviation. Here we only subtract the mean: all pixels measure speed in the same units, and we want regions with larger physical fluctuations to contribute more. Standardizing each pixel would give weakly and strongly fluctuating regions equal variance.

In [3]:
data_mean = ...  # TODO: mean over snapshots
data_centered = ...  # TODO: subtract the mean field

In [ ]:
# Compare a snapshot, the mean field, and their difference.
speed_limits = (
    float(min(data[0].min(), data_mean.min())),
    float(max(data[0].max(), data_mean.max())),
)
fig, axes = plt.subplots(1, 3, figsize=(15, 3), layout="constrained")
image = show_field(axes[0], data[0], "First snapshot", limits=speed_limits)
show_field(axes[1], data_mean, "Mean field", limits=speed_limits)
fig.colorbar(image, ax=axes[:2], label="Speed", shrink=0.8)
# TODO: select the first snapshot after centering.
image = show_field(axes[2], ..., "Snapshot minus mean", signed=True)
fig.colorbar(image, ax=axes[2], label="Speed difference", shrink=0.8)
plt.show()

What properties of the flow are already present in the mean field? Where do the departures from the mean appear largest?

### 2. Build PCA with SVD (core)

We will first implement the PCA steps ourselves using NumPy's SVD, then check the result against scikit-learn in Section 4.

**2.a)** Compute the SVD of the centered data:
$$X_c=U\Sigma V^\top.$$
The notes use the full SVD: $U$ is $T\times T$, $\Sigma$ is $T\times d$, and $V^\top$ is $d\times d$. Here $T\ll d$, so we can omit the zero columns of $\Sigma$ and the corresponding rows of $V^\top$ without changing the reconstructed matrix. In this reduced SVD, $\Sigma$ is $T\times T$ and $V^\top$ is $T\times d$.

Use `np.linalg.svd` with **`full_matrices=False`**. It returns `U`, a vector `s` of singular values in decreasing order, and `Vt`, with shapes $(T,T)$, $(T,)$, and $(T,d)$. **Keep `full_matrices=False`: with $d=102\,212$, the full $d\times d$ matrix alone would occupy about 84 GB in double precision!**

The rows of `Vt` are the spatial principal components $\mathbf{v}_i$. Why do we use the right singular vectors rather than the left ones?

In [ ]:
U, s, Vt = ...  # TODO: reduced SVD of the centered data

**2.b)** Complete the code below to display the first four principal components. Then try other indices in `component_indices` (Python indices start at zero). Unlike a speed image, a component can have positive and negative values. What does multiplying one of these patterns by a positive or negative coefficient do?

The overall sign of a component is arbitrary: changing the signs of both the component and its coefficient leaves the reconstruction unchanged.

In [ ]:
component_indices = [0, 1, 2, 3]  # Choose four component indices to display.
fig, axes = plt.subplots(2, 2, figsize=(12, 6), layout="constrained")
for i, ax in zip(component_indices, axes.flat):
    # TODO: select the spatial pattern for component index i.
    image = show_field(ax, ..., f"Component {i + 1}", signed=True)
    fig.colorbar(image, ax=ax, label="Component value", shrink=0.8)
plt.show()

### 3. Choose how many components to retain (core)

**3.a)** For the covariance matrix $C=X_c^\top X_c/T$, obtain the component variances from the singular values, then compute the fraction of total variance explained by each component:
$$\lambda_i=\frac{\sigma_i^2}{T},\qquad r_i=\frac{\lambda_i}{\sum_j\lambda_j}.$$
Compute the cumulative explained variance and the **smallest number of components** that retains at least 90% of the variance.

*Hint:* Use `np.cumsum`. The first array entry corresponds to **one** component, not zero.

The notes discuss choosing unusually large eigenvalues separated from the bulk. Here our aim is reconstruction: 90% is a chosen accuracy target, not a unique number of physical modes.

In [ ]:
component_variances = ...  # TODO: compute from s
variance_ratio = ...  # TODO: normalize by total variance
cumulative_variance = ...  # TODO: cumulative explained variance ratio
k90 = ...  # TODO: first component count reaching 0.90

In [ ]:
counts = ...  # TODO: component counts from 1 to the number of components
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(counts, ...)  # TODO: plot the total variance fraction retained
ax.axhline(0.90, color="gray", linestyle="--")
ax.axvline(k90, color="gray", linestyle="--")
ax.set(xlabel="Number of components", ylabel="Cumulative explained variance", ylim=(0, 1.02))
plt.show()
print(f"{k90} components retain at least 90% of the variance.")

How much variance do 10 and 50 components retain? Does the answer suggest that just one or two patterns capture all the changing flow structures?

In [ ]:
# Provided: read off the two comparison values.
for k in [10, 50]:
    print(f"{k} components: {100 * cumulative_variance[k - 1]:.1f}% of the variance")

### 4. Project and reconstruct the snapshots (core)

**4.a)** Compute the coefficients by projecting the centered snapshots onto the principal directions using matrix multiplication:
$$A=X_cV=U\Sigma,\qquad A_{ti}=a_{i,t}.$$
Use `data_centered` and `Vt` to obtain `coefficients_svd`. Check the matrix shapes to decide where a transpose is needed. Then compute the same coefficients from `U` and `s`; the provided check compares the two expressions.

Reconstruct the first snapshot using its first ten coefficients:
$$\widehat{\mathbf{x}}_t^{(k)}=\bar{\mathbf{x}}+\sum_{i=1}^{k}a_{i,t}\mathbf{v}_i.$$
Use `coefficients_svd` and `Vt`, and remember the mean field. After reconstructing with ten components, try `k = 50` and `k = 90`. What changes in the reconstructed wake?

In [ ]:
coefficients_svd = ...  # TODO: project using data_centered and Vt
coefficients_from_us = ...  # TODO: compute U Sigma using U and s
coefficient_error = np.linalg.norm(coefficients_svd - coefficients_from_us) / np.linalg.norm(coefficients_svd)
print(f"Relative difference between coefficient formulas: {coefficient_error:.2e}")

k = 10
reconstructed = ...  # TODO: reconstruct snapshot 0 using k components

In [ ]:
# Provided: use identical color limits to compare the images fairly.
limits = (
    float(min(data[0].min(), data_mean.min(), reconstructed.min())),
    float(max(data[0].max(), data_mean.max(), reconstructed.max())),
)
fig, axes = plt.subplots(1, 3, figsize=(15, 3), layout="constrained")
for ax, values, title in zip(
    # TODO: select the original snapshot corresponding to the reconstruction.
    axes, [..., data_mean, reconstructed], ["Original", "Mean only", f"{k} components"]
):
    image = show_field(ax, values, title, limits=limits)
fig.colorbar(image, ax=axes, label="Speed", shrink=0.8)
plt.show()

**4.b) Reconstruct a batch of snapshots.** Write `reconstruct_snapshots` to reconstruct all rows of a coefficient matrix using the first `k` components and the mean. It should also work for `k=0`, returning the mean field for every snapshot.

Then implement the relative reconstruction error
$$E(k)=\frac{\sum_t\|\mathbf{x}_t-\widehat{\mathbf{x}}_t^{(k)}\|^2}
{\sum_t\|\mathbf{x}_t-\bar{\mathbf{x}}\|^2}.$$
Sum over both snapshots and pixels. The denominator measures the variation in the original data, so $E(0)=1$.

In [ ]:
def reconstruct_snapshots(coefficients, components, mean, k):
    # coefficients: one row per snapshot; components: one row per component.
    ...  # TODO: return the reconstructed snapshots

def relative_reconstruction_error(original, reconstructed, mean):
    ...  # TODO: return total squared error / total variation around the mean

**4.c) Measure the error as we retain more components.** For each component count below, use your functions to reconstruct the original dataset and compute $E(k)$. Store only the errors; there is no need to keep every reconstructed dataset in memory.

Compare the measured errors with the prediction from the explained-variance fractions:
$$E(k)=1-\sum_{i=1}^{k}r_i.$$
Use `variance_ratio` to compute the prediction. The same expression should work for `k=0` by summing an empty slice.

Why must the error decrease as more components are retained? What should happen when all components are retained?

In [ ]:
reconstruction_counts = np.unique([0, 1, 2, 5, 10, 20, 50, k90, len(s)])
reconstruction_errors = []
predicted_errors = []

for n_components in reconstruction_counts:
    # TODO: reconstruct with your function, using coefficients_svd and Vt.
    reconstructed_batch = ...
    # TODO: compute and store the measured relative error.
    reconstruction_errors.append(...)
    # TODO: compute and store the unexplained-variance fraction.
    predicted_errors.append(...)

In [ ]:
# Compare the measured errors with the variance prediction.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(reconstruction_counts, ..., color="tab:blue", label="Unexplained variance")  # TODO: predicted errors
ax.plot(reconstruction_counts, ..., "o", color="tab:blue", label="Measured reconstruction error")  # TODO: measured errors
ax.set(xlabel="Number of components", ylabel="Relative reconstruction error", ylim=(-0.02, 1.05))
ax.legend()
plt.show()
print("Measured and predicted errors agree:",
      np.allclose(reconstruction_errors, predicted_errors, rtol=1e-6, atol=1e-10))

**4.d) Do all snapshots have the same reconstruction quality?** Reconstruct the dataset with `k90` components. Compute the relative error of each snapshot separately, now summing over pixels only:
$$e_t=\frac{\|\mathbf{x}_t-\widehat{\mathbf{x}}_t^{(k_{90})}\|^2}
{\|\mathbf{x}_t-\bar{\mathbf{x}}\|^2}.$$
Does retaining 90% of the total variance guarantee that every snapshot has a relative error below 10%?

In [ ]:
reconstructed_at_k90 = ...  # TODO: reconstruct all snapshots with your function
squared_error_per_snapshot = ...  # TODO: sum squared residuals over pixels
variation_per_snapshot = ...  # TODO: sum squared deviations from the mean over pixels
relative_error_per_snapshot = ...  # TODO: normalize each snapshot's error

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
# TODO: choose the relative-error array with one entry per snapshot.
ax.plot(np.arange(nt), ..., ".", label=f"{k90} components")
ax.axhline(0.10, color="gray", linestyle="--", label="10% relative error")
ax.set(xlabel="Snapshot index", ylabel="Relative reconstruction error")
ax.legend()
plt.show()
print(f"Individual relative errors range from {relative_error_per_snapshot.min():.1%} "
      f"to {relative_error_per_snapshot.max():.1%}.")

**4.e) Check against scikit-learn.** The cell below fits `PCA` to the original data; the library subtracts the mean internally. Compare its singular values, variance fractions, and reconstruction with your results.

Components may have opposite signs in the two implementations. We compare the absolute dot products of the first ten pairs of unit vectors: values close to one mean the directions agree. Relative differences close to zero indicate agreement up to floating-point precision.

From here on, we use the fitted `pca` model and its `coefficients` for the remaining analyses.

In [ ]:
# Provided: an independent check of the PCA steps implemented above.
pca = PCA(svd_solver="full")
pca.fit(data)
coefficients = pca.transform(data)
reconstructed_sklearn = coefficients[0, :k] @ pca.components_[:k] + pca.mean_

singular_value_error = np.linalg.norm(s - pca.singular_values_) / np.linalg.norm(s)
print(f"Relative difference in singular values: {singular_value_error:.2e}")
print("Variance fractions agree:",
      np.allclose(variance_ratio, pca.explained_variance_ratio_, rtol=1e-4, atol=1e-7))
direction_agreement = np.abs(np.sum(Vt[:10] * pca.components_[:10], axis=1))
print("Agreement of first ten directions:", np.round(direction_agreement, 5))
print("Reconstructions agree:",
      np.allclose(reconstructed, reconstructed_sklearn, rtol=1e-4, atol=1e-5))

### 5. Identify a dominant source of variation (core)

When data contain several sources of variation, PCA can help identify a dominant one. Suppose an external disturbance affects part of our recording. Can we identify its spatial pattern and remove its contribution?

We add a smooth pattern $\mathbf{p}$ with a random amplitude $b_t$ to snapshots 50–149:
$$\mathbf{y}_t=\mathbf{x}_t+b_t\mathbf{p}.$$
The amplitudes have zero mean, so the average field stays unchanged, but the disturbance adds variation. The pattern increases speed on one side of the foil and decreases it on the other. This is a simple model of an external gust in our **speed measurements**, not a new fluid simulation.

**5.a)** Run the setup and compare an original snapshot with its perturbed version. Is the disturbance easy to notice?

In [ ]:
from scipy.ndimage import distance_transform_edt

# A broad pattern that fades near the foil, normalized to unit spatial RMS.
perturb_rows, perturb_cols = np.indices(foil_grid.shape)
perturb_mid = np.where(foil_grid)[0].mean()
perturb_grid = np.sin(np.pi * (perturb_rows - perturb_mid) / (NY / 2))
perturb_grid *= np.exp(-0.5 * ((perturb_cols - 0.6 * NX) / (0.45 * NX)) ** 2)
perturb_grid *= 1 - np.exp(-(distance_transform_edt(~foil_grid) / 30) ** 4)
perturb_pattern = perturb_grid[~foil_grid].astype(np.float32)
perturb_pattern /= np.sqrt(np.mean(perturb_pattern ** 2))

strength = 0.04  # Typical per-pixel change: about 4% of the mean speed.
perturb_active = np.zeros(nt, dtype=bool)
perturb_active[50:150] = True
perturb_rng = np.random.default_rng(0)
perturb_draws = perturb_rng.normal(size=perturb_active.sum())
perturb_draws -= perturb_draws.mean()
perturb_draws /= np.sqrt(np.mean(perturb_draws ** 2))
perturb_amplitude = np.zeros(nt, dtype=np.float32)
perturb_amplitude[perturb_active] = strength * perturb_draws
data_perturbed = data + perturb_amplitude[:, None] * perturb_pattern

# Select a typical disturbance and the strongest one for the comparisons.
perturb_typical = int(np.where(perturb_active)[0][
    np.argmin(np.abs(np.abs(perturb_draws) - 1))
])
perturb_strongest = int(np.argmax(np.abs(perturb_amplitude)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3), layout="constrained")
perturb_fields = [data[perturb_typical], ...]  # TODO: select the same snapshot after perturbation
perturb_limits = (min(x.min() for x in perturb_fields), max(x.max() for x in perturb_fields))
for ax, values, title in zip(axes, perturb_fields, ["Original", "With disturbance"]):
    image = show_field(ax, values, title, limits=perturb_limits)
fig.colorbar(image, ax=axes, label="Speed", shrink=0.8)
plt.show()

**5.b)** Fit a new PCA model to `data_perturbed` and compute its coefficients. The plots below compare the first component with the injected pattern and show each snapshot in the first two PC coordinates.

How does the disturbance appear in PC space? What does moving along PC1 mean for the speed field?

In [ ]:
pca_perturbed = PCA(svd_solver="full")
pca_perturbed.fit(...)  # TODO: fit to the perturbed snapshots
coefficients_perturbed = ...  # TODO: project the perturbed snapshots

In [ ]:
# Choose PC1's sign only for easier visual comparison with the known pattern.
if pca_perturbed.components_[0] @ perturb_pattern < 0:
    pca_perturbed.components_[0] *= -1
    coefficients_perturbed[:, 0] *= -1

fig, axes = plt.subplots(2, 2, figsize=(12, 7), layout="constrained")
for ax, values, title in zip(
    axes[0], [perturb_pattern, pca_perturbed.components_[0]],
    ["Injected pattern", "First principal component"],
):
    image = show_field(ax, values / np.max(np.abs(values)), title,
                       signed=True, limits=(-1, 1))
fig.colorbar(image, ax=axes[0], label="Pattern (rescaled for comparison)", shrink=0.8)

for flag, label in [(~perturb_active, "No disturbance"), (perturb_active, "Disturbed interval")]:
    axes[1, 0].scatter(
        ...,  # TODO: PC1 coordinates of the snapshots selected by flag
        ...,  # TODO: PC2 coordinates of those same snapshots
        s=18, label=label,
    )
axes[1, 0].set(xlabel="PC1 coefficient", ylabel="PC2 coefficient")
axes[1, 0].legend()

# Divide by the pattern's norm to compare coefficients with injected amplitudes.
axes[1, 1].plot(perturb_amplitude, label="Injected amplitude", linewidth=1)
axes[1, 1].plot(coefficients_perturbed[:, 0] / np.linalg.norm(perturb_pattern),
                label="Estimated from PC1", linewidth=1)
axes[1, 1].set(xlabel="Snapshot index", ylabel="Amplitude")
axes[1, 1].legend()
plt.show()
print(f"PC1 explains {100 * pca_perturbed.explained_variance_ratio_[0]:.1f}% of the variance.")

Usually, PCA denoising keeps the leading components and discards smaller ones. Here we consider a different situation: a strong external disturbance. We remove the first component because we recognize its pattern as unwanted, not simply because it explains a lot of variance.

**5.c)** We recognize the first component as the unwanted disturbance. Remove its contribution from every perturbed snapshot:
$$\mathbf{x}^{\mathrm{cleaned}}_t=\mathbf{y}_t-a_{1,t}\mathbf{v}_1,$$
where $a_{1,t}$ and $\mathbf{v}_1$ now come from the PCA fitted to the perturbed data.

*Hint:* `coefficients_perturbed[:, :1] @ pca_perturbed.components_[:1]` gives the contribution of PC1 to every snapshot. Does subtracting this contribution remove the mean field?

In [ ]:
data_cleaned = ...  # TODO: subtract the contribution of PC1

In [ ]:
# Compare a typical disturbance (top) and the strongest one (bottom).
fig, axes = plt.subplots(2, 3, figsize=(14, 5), layout="constrained")
for row, t in enumerate([perturb_typical, perturb_strongest]):
    perturb_fields = [data[t], data_perturbed[t], ...]  # TODO: the same snapshot after cleaning
    perturb_limits = (min(x.min() for x in perturb_fields), max(x.max() for x in perturb_fields))
    for ax, values, title in zip(axes[row], perturb_fields, ["Original", "With disturbance", "After removing PC1"]):
        image = show_field(ax, values, f"{title}, snapshot {t}", limits=perturb_limits)
        ax.set_axis_off()
    fig.colorbar(image, ax=axes[row], label="Speed", shrink=0.8)
plt.show()

perturb_error_before = np.mean((data_perturbed - data) ** 2)
perturb_error_after = np.mean((data_cleaned - data) ** 2)
print(f"Mean squared error: {perturb_error_before:.3g} before, {perturb_error_after:.3g} after cleaning.")

How well did we recover the original flow? Why might the recovery be imperfect?

PCA finds directions of large variation; it does not decide which variations are useful. Here our knowledge of the injected disturbance tells us which component to remove.

### Extensions

Extensions A–C use the original data and PCA model from Sections 1–4.

#### A. Component variances

Plot the component variances below. Are the largest values clearly separated from the rest? You need not find a sharp separation for PCA to be useful.

Implementation detail: the notes use $X_c^\top X_c$, the covariance above includes $1/T$, and sklearn reports variances using $1/(T-1)$. These overall factors do not change the principal directions or explained-variance ratios.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
# TODO: plot the individual component variances reported by pca (not variance fractions).
ax.plot(counts, ..., ".", markersize=3)
ax.set(xlabel="Component number", ylabel="Component variance")
plt.show()

#### B. Compare several snapshots

Complete the comparison for three reproducibly selected snapshots. Compare the original with the mean alone, then with 1, 10, 50, and your `k90` components. Which details appear as the number of components increases?

In [ ]:
# Provided: reconstruct only the three snapshots we want to display.
rng = np.random.default_rng(0)
selected = rng.choice(nt, size=3, replace=False)
component_counts = list(dict.fromkeys([0, 1, 10, 50, k90]))
selected_coefficients = coefficients[selected]
reconstructions = {
    n: selected_coefficients[:, :n] @ pca.components_[:n] + data_mean
    for n in component_counts
}
print("Selected snapshot indices:", selected)

In [ ]:
fig, axes = plt.subplots(
    len(component_counts) + 1, len(selected),
    figsize=(12, 1.7 * (len(component_counts) + 1)), layout="constrained",
)
for col, t in enumerate(selected):
    # TODO: select this snapshot from each reconstructed batch.
    # reconstructions[n] contains only the three selected snapshots, in order.
    fields = [data[t]] + [... for n in component_counts]
    limits = (float(min(x.min() for x in fields)), float(max(x.max() for x in fields)))
    titles = [f"Original, snapshot {t}"] + [
        "Mean only" if n == 0 else ("1 component" if n == 1 else f"{n} components") for n in component_counts
    ]
    for row, (values, title) in enumerate(zip(fields, titles)):
        image = show_field(axes[row, col], values, title, limits=limits)
        axes[row, col].set_axis_off()
    fig.colorbar(image, ax=axes[:, col], label="Speed", shrink=0.25)
plt.show()

#### C. Does PCA predict the future?

Complete the plot of the first coefficient in the order of the supplied snapshots. Then change `component_index` to inspect another component. The dataset does not include physical timestamps, so the horizontal axis is the snapshot index.

Does PCA itself tell us how to obtain the next coefficient? Is an irregular-looking curve enough to decide whether forecasting is possible?

In [ ]:
component_index = 0  # Change this to inspect another PC.
fig, ax = plt.subplots(figsize=(9, 3))
# TODO: select one coefficient for every snapshot, using component_index.
ax.plot(np.arange(nt), ..., linewidth=1, label=f"PC{component_index + 1} coefficient")
ax.set(xlabel="Snapshot index", ylabel="Coefficient", title="PCA coefficient across snapshots")
ax.legend()
plt.show()

#### D. A weaker disturbance

Try `strength = 0.01` and `0.02` in Section 5, keeping the random seed fixed, and rerun that section. Does PC1 still resemble the injected pattern? Does removing it always improve the data?

### References

[1] Towne, A., Yeh, C., Patel, H., Taira, K. (2022). *Turbulent airfoil wake large eddy simulation*. University of Michigan, Deep Blue Data. https://doi.org/10.7302/0e3g-6j84

[2] Taira, K. et al. (2017). *Modal Analysis of Fluid Flows: An Overview*. https://doi.org/10.48550/arXiv.1702.01453

PCA is closely related to proper orthogonal decomposition (POD) in fluid mechanics [2]. Here we analyze the supplied speed fields.